In [1]:
import os
import math
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn.functional as F  
from torch.utils.tensorboard import SummaryWriter

from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADVisualizer   import CADVisualizer
from Pred_NN_Classes import PPNet
from Decoder_CLasses.VoronoiDecorder import VoronoiDecoder,VoronoiModelVisualizer
from Training.MainTrain import TrainingConfig, NN_Trainer
from neuraltomo_fem import run_fem_loss
from problems.ThickenShell import ThickenShell

import pyvista as pv


# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)


if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
# -------- PYVISTA BACKEND --------
def setup_pyvista(device):
    is_mac = sys.platform == "darwin"

    # Mac + MPS: prefer static to avoid VTK/trame hangs
    if is_mac :
        pv.OFF_SCREEN = True
        pv.set_jupyter_backend("static")
        backend = "static"
    else:
        try:
            pv.set_jupyter_backend("trame")
            backend = "trame"
        except Exception:
            pv.OFF_SCREEN = True
            pv.set_jupyter_backend("static")
            backend = "static"

    print(f"PyVista backend: {backend}")

setup_pyvista(device)


gmesh was loaded successfully!
Code Directory: /home/arash/HDV_Shell
Test Step files Directory: /home/arash/HDV_Shell/Testparts
device: cuda
PyVista backend: trame


In [8]:
viz = CADVisualizer()
# Laoding model and extracting mesh and tensors as input
FreeFormSurf1  = TesPartsDir / "FreeFormCrv1.stp"
FreeFormSurf2A = TesPartsDir / "FreeFormSurf2A.STEP"
FreeFormSurf3 = TesPartsDir / "FreeForm3.stp"
ConeTaped = TesPartsDir / "ConeTaped.stp"
FreeFormCLosed = TesPartsDir / "FreeFormClosed.stp"
Planar = TesPartsDir / "Planar.stp"
YachtBodypart  = TesPartsDir / "YachtBodypart.stp"
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Cube           = TesPartsDir / "Cube.stp"
CircularSur2   = TesPartsDir / "CircularSur2.stp"
Conic          = TesPartsDir / "Conic.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
FullCylinder   = TesPartsDir / "FullCylinder.stp"
Sphere         = TesPartsDir / "Sphere.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Tidebottle     = TesPartsDir / "Tidebottle.STEP"


shape_path = Cube

# Meshing controls
# OCC meshing (global fallback for mode="mesh"):
#   - deflection: smaller -> finer OCC mesh
#   - angle: smaller -> finer OCC mesh on curved regions
# Manual UV-grid meshing (used when freeform_mesher="manual" on supported analytic faces):
#   - n_u, n_v: more samples -> finer UV grid mesh
# Gmsh meshing (used for free-form faces when freeform_mesher is "auto" or "gmsh"):
#   - gmsh_size_scale: smaller -> finer mesh, larger -> coarser mesh
#   - gmsh_algorithm: picks the Gmsh 2D meshing algorithm (6 is a solid default)
# Mesher selection:
#   - freeform_mesher="auto": try Gmsh first, then fall back to OpenCascade
#   - freeform_mesher="gmsh": require Gmsh for free-form faces
#   - freeform_mesher="manual": use UV-grid on supported analytic faces, otherwise OpenCascade

Case_name = shape_path.stem
print(Case_name)
generator = CADTensorGenerator(
    deflection=0.001,
    angle=0.001,
    metric_tol=1e-9,
    det_min=1e-5,
    n_u=30  ,
    n_v=30,
    freeform_mesher="auto",   #"auto"
    gmsh_size_scale=0.5,
    gmsh_algorithm=6,
    device=device,
    selected_face_index=3,
)

mesh_df, faces_df, tensors = generator.generate_from_file(
    shape_path=str(shape_path),
    input_ring=1,
    mode="mesh", #"1: mesh" "2:Sampled_points "
    M_per_face=2000,
    pool_size_factor=10,
    fps_pool_factor=4,
    use_fps=True,
    triangulation_max_edge_rel=0.1,
)
uv = tensors["uv"]
Xu = tensors["Xu"]
Xv = tensors["Xv"]
points_xyz = tensors["points_xyz"]
face_areas = tensors["face_areas"]
faces_ijk = tensors["faces_ijk"]
face_id = tensors["face_id"]
boundary_idx_ring1 = tensors["boundary_idx_ring1"]
pv_faces = tensors["pv_faces"]

face_tensor = tensors["face_tensors"][0]
face_u_periodic = bool(face_tensor["u_periodic"])
face_v_periodic = bool(face_tensor["v_periodic"])

bbx_all = list(tensors["BBX"].values())
xmin = min(b["xmin"] for b in bbx_all)
xmax = max(b["xmax"] for b in bbx_all)
ymin = min(b["ymin"] for b in bbx_all)
ymax = max(b["ymax"] for b in bbx_all)
zmin = min(b["zmin"] for b in bbx_all)
zmax = max(b["zmax"] for b in bbx_all)

dx = xmax - xmin
dy = ymax - ymin
dz = zmax - zmin

print(f"Selected CAD face index: {generator.selected_face_index}")
print(f"Number of faces used: {tensors['num_faces']}")
print(f"Number of sampled points: {uv.shape[0]}")
print(f"Selected-face BBX dimensions: dx={dx:.4f}, dy={dy:.4f}, dz={dz:.4f}")

if face_u_periodic:
    print("Face is periodic in u direction")
if face_v_periodic:
    print("Face is periodic in v direction")

viz.visualize_show_Model(points_xyz, pv_faces)

pts = points_xyz.detach().cpu().numpy()
cloud = pv.PolyData(pts)
plotter = pv.Plotter()
plotter.add_mesh(cloud, render_points_as_spheres=True, point_size=6)
plotter.show()


Cube
Model is meshed using Gmsh tool
MinVolFrac: 0.08225081861019135
Mesh edge length: min=0.021431 max=0.0405388 median=0.0333333
Selected CAD face index: 3
Number of faces used: 1
Number of sampled points: 4342
Selected-face BBX dimensions: dx=0.0000, dy=2.0000, dz=2.0000


Widget(value='<iframe src="http://localhost:33859/index.html?ui=P_0x785b2c491c00_11&reconnect=auto" class="pyv…

Widget(value='<iframe src="http://localhost:33859/index.html?ui=P_0x785b2c492740_12&reconnect=auto" class="pyv…

In [9]:

Include_FEM= True
LoadingCase = "Tensile in X direction"

voxel_size = (dx+dy+dz)/40
fixed_height_shell= voxel_size*2
tangential_tol = voxel_size*0.8

if(Include_FEM):
    shell_problem = ThickenShell(
        thickness=fixed_height_shell,
        BC_dir = "z",
        Load_magnitude=0.1,
        voxel_size=voxel_size,
        extra_layers=1,
        tensors=tensors,
        tangential_tol=tangential_tol,
        load_case="tensile_compression",
    )
    # shell_problem =None
    # fem =None
    fem = run_fem_loss.NeuralTOMOFEM(shell_problem, device=device, isotropic=False)
    shell_problem.debug_voxel_stats()
    loading_img = shell_problem.show_voxels_surface_and_bc(
        return_img=True,
        off_screen=True,
        window_size=(560, 430),
        show=True,
    )
else:
    shell_problem = None
    fem = None
    loading_img = None   

=== Voxel Stats ===
brep_bbox: {'xmin': -1.0, 'xmax': -1.0, 'ymin': -1.0, 'ymax': 1.0, 'zmin': -1.0, 'zmax': 1.0}
mesh: {'nelx': 4, 'nely': 24, 'nelz': 24, 'elemSize': array([0.1, 0.1, 0.1]), 'type': 'grid'}
elem_centers shape: (24, 4, 24, 3)
node_coords shape: (25, 5, 25, 3)
occupied voxels: 968
total voxels: 2304
occupancy ratio: 0.4201388888888889
voxels with assigned surface samples: 484
occupied voxels with assigned surface samples: 484
voxelized volume: 0.9680000000000002
thickness: 0.2
voxel_size: 0.1
target approx volume (sum(face_areas)*thickness): 0.8
volume ratio voxel/target: 1.2100000000000002


Widget(value='<iframe src="http://localhost:33859/index.html?ui=P_0x7858cc3c0b20_13&reconnect=auto" class="pyv…

In [ ]:
# ============================================================
# Important tuning notes
# ============================================================
#
# duplicate_merge_sigma
#   Controls when two seeds are considered duplicates.
#   Larger value -> seeds deactivate/merge more easily.
#   Smaller value -> more seeds stay active.
#   For 10 seeds, 0.03-0.05 is safer than 0.1.
#
# lam_rep
#   Seed repulsion loss weight.
#   Larger value -> seeds spread apart more, fewer collapses.
#   Too large -> seeds may ignore FEM optimum.
#   Good range: 0.02-0.10.
#
# allow_seed_outside_domain
#   If True, seeds can leave [0,1] UV and become inactive.
#   If False, seeds are clamped inside the domain.
#   For stable active seed count, use False.
#
# min_active_seeds
#   Minimum active seeds required for accepting a best result.
#   Larger value -> rejects collapsed solutions.
#   For 10 seeds, use 8 or 9.
#
# use_band_weighted_fiber_pairs
#   If True, fiber direction follows visible Voronoi strut bands.
#   If False, fiber direction uses smoother global seed ownership.
#   For your case, True is better.
#
# fiber_band_prior_power
#   Strength of local band preference in fiber direction.
#   Larger value -> fibers lock more strongly to visible struts.
#   Smaller value -> smoother, more global fiber field.
#   Good range: 1.0-4.0.
#
# fiber_band_prior_floor
#   Minimum fallback weight for nonlocal soft pair information.
#   Smaller value -> cleaner local strut alignment.
#   Larger value -> smoother near junctions/endpoints.
#   Good range: 0.01-0.10.
#
# width_target_frac_max
#   Maximum target width ratio allowed by width regularization.
#   Larger value -> bars can become very thick when seeds deactivate.
#   Smaller value -> keeps struts narrower.
#   Good range: 0.45-0.65.
#
# collapse_min_seed_dist_factor
#   Collapse detector threshold in units of w_min.
#   Larger value -> detects seed collapse earlier.
#   Too large -> frequent restore events.
#   Good range: 3-6.
#
# anchor_guard_min_seed_dist_factor
#   Minimum spacing required before updating rolling seed anchors.
#   Larger value -> anchors update only when seeds are well separated.
#   Good range: 2-4.
#
# post_restore_seed_freeze_steps
#   Number of steps to freeze/restrict seed motion after collapse restore.
#   Larger value -> more stable recovery.
#   Too large -> slower exploration.
#   Good range: 50-150.
#
# post_restore_offset_scale
#   Seed movement scale immediately after restore.
#   Smaller value -> gentler recovery.
#   Larger value -> faster but less stable recovery.
#   Good range: 0.1-0.5.
#
# project_seed_spacing_each_step
#   If True, seeds are projected apart every step.
#   Helps prevent collapse, but can constrain optimization.
#   Use True for this case.
#
# seed_projection_iters
#   Number of spacing projection iterations per step.
#   Larger value -> stronger separation enforcement.
#   Good range: 2-6.


In [10]:
Explanation = "_ZX_Tensile"

cfg = TrainingConfig(
    LoadingCasee=LoadingCase,
    seed_number=20,
    use_Metric_anisotropy=False,
    fixed_height=fixed_height_shell,
    target_volfrac=0.15,

    seed_repulsion_sigma=0.1,
    boundary_margin=0.05,

    num_steps=2000,
    log_every=100,
    early_stop_start=10000,
    patience=200,
    min_delta=1e-10,

    freeze_w=False,
    w_const=0.12,
    w_min=0.003,
    w_max_ratio=0.25,
    duplicate_merge_sigma=0.05,

    use_band_weighted_fiber_pairs=True,
    fiber_band_prior_power=2.0,
    fiber_band_prior_floor=0.02,

    use_boundary_attachment=True,
    predict_boundary_params=False,
    boundary_volume_assist=0.0,
    boundary_attach_width=0.001,
    boundary_attach_beta=0.003,
    boundary_attach_alpha=1.0,
    boundary_attach_width_min=5e-6,
    boundary_attach_width_max=5e-5,
    boundary_attach_alpha_min=0.05,
    boundary_attach_alpha_max=1.0,
    boundary_attach_beta_min=0.003,
    boundary_attach_beta_max=0.05,

    predict_tau=False,
    tau=0.02,
    tau_pred_start=0.1,
    tau_pred_min=1e-3,
    tau_pred_max=1.0,
    tau_anneal_final=0.03,
    tau__anneal_final=0.03,
    tau_anneal_start_frac=0.0,
    tau_anneal_ramp_frac=0.5,
    beta=0.02,

    density_projection_strength=0.45,
    density_projection_threshold=0.35,
    density_projection_gamma=0.06,

    use_rolling_seed_anchors=True,
    seed_anchor_momentum=0.08,
    seed_anchor_warmup_frac=0.005,
    guard_seed_anchor_updates=False,
    anchor_guard_rep_max=0.30,
    anchor_guard_bnd_max=0.80,
    anchor_guard_vol_eff_min=0.10,
    anchor_guard_width_factor_min=1.20,
    anchor_guard_min_seed_dist_factor=2.0,

    collapse_min_seed_dist_factor=2,
    project_seed_spacing_each_step=True,
    seed_projection_iters=4,

    lam_fem=10,
    lam_vol=60,
    lam_rep=0,
    lam_bnd=0,
    lam_strut=0.0,
    lam_strut_edge=1.0,
    lam_strut_void=3.0,
    lam_width_active=0,


    width_target_frac=0.04,
    width_target_sparse_boost=1.5,
    width_target_frac_max=0.12,
    width_warmup_start_frac=0.1,
    width_warmup_ramp_frac=0.20,
    lam_width_active_hard_multiplier=8.0,
    decoder_raw_temp=1.25,
    w_head_bias_init=None,

    lam_seed_active=2.0,
    min_active_seeds=5,

    comp_normalize_by=None,
    normalize_losses=False,
    fem_density_floor=0.005,
    skip_bad_fem_steps=True,

    allow_seed_outside_domain=True,
    allow_seed_outside_domain_warmup_frac=0.05,
    seed_domain_margin=0.35,

    hard_refine_start_frac=0.95,
    freeze_tau_head_during_hard_refine=True,
    hard_refine_width_multiplier=1.0,

    use_boundary_weighted_volume=False,
    boundary_vol_weight=0.2,
    effective_volume_power=2.0,
    lam_vol_effective=4,
    lam_vol_sharp=1,
    sharp_vol_start_frac=0.6,
    sharp_vol_ramp_frac=0.3,

    lr_seed_refine=1e-4,
    lr_delta_head=1e-4,
    lr_mlp=1e-4,
    lr_w_head=5e-5,
    lr_h_head=1e-4,
    lr_boundary_heads=1e-4,

    eps=1e-12,
    Offset_scale=1.0,
    scheduler_milestones=(0.75, 0.9),
    scheduler_gamma=0.2,
    save_fem_debug_history=True,
    grad_clip_norm=1.0,

    tensorboard_enabled=True,
    tensorboard_log_root="runs",
    experiment_name=f"{Case_name}{Explanation}",
    tb_flush_secs=10,
    tb_log_histograms_every=100,

    MakeTimelaps=True,
    timelapse_output_folder=f"Case_Studies/{Case_name}{Explanation}",
    timelapse_frame_step=10,
    TM_laps_res_u=100,
    TM_laps_res_v=100,
    TM_laps_Thr=0.3,
)


trainer = NN_Trainer(
    generator=generator,
    viz=viz,
    decoder_cls=VoronoiDecoder,
    ppnet_cls=PPNet,
    fem=fem,
    shell_problem=shell_problem,
    config=cfg,
    loading_img=loading_img,
)

print(cfg)

result = trainer.train(
    shape_path,
    face_tensors=tensors["face_tensors"],
)
trainer.visualize_result_stepwise(result, points_xyz, faces_ijk)

TensorBoard log dir: runs/Cube_ZX_Tensile
TrainingConfig(seed_number=20, training_face_index=0, LoadingCasee='Tensile in X direction', use_Metric_anisotropy=False, fixed_height=0.2, target_volfrac=0.15, seed_repulsion_sigma=0.1, boundary_margin=0.05, freeze_w=False, use_boundary_attachment=True, boundary_volume_assist=0.0, w_const=0.12, boundary_attach_width=0.001, boundary_attach_beta=0.003, boundary_attach_alpha=1.0, hollow_void_threshold=0.85, hollow_edge_threshold=0.45, hollow_temp=0.05, hollow_rho_edge_min=0.75, boundary_attach_width_min=5e-06, boundary_attach_width_max=5e-05, duplicate_merge_sigma=0.05, boundary_attach_alpha_min=0.05, boundary_attach_alpha_max=1.0, boundary_attach_beta_min=0.003, boundary_attach_beta_max=0.05, predict_tau=False, w_min=0.003, w_max_ratio=0.25, lam_fem=10, lam_vol=60, lam_rep=0, lam_bnd=0, lam_strut=0.0, lam_strut_edge=1.0, lam_strut_void=3.0, lam_width_active=0, lam_seed_active=2.0, width_target_frac=0.04, width_target_sparse_boost=1.5, width_targ

Training:   0%|          | 0/2000 [00:00<?, ?it/s]

[step 0] Non-finite gradients detected, optimizer step skipped. Examples: face=0:global_latent, face=0:seed_identity.embedding.weight, face=0:seed_refiner.seed_refine.0.weight, face=0:seed_refiner.seed_refine.0.bias, face=0:seed_refiner.seed_refine.2.weight, face=0:seed_refiner.seed_refine.2.bias, face=0:seed_refiner.delta_head.weight, face=0:seed_refiner.delta_head.bias
New best_step=0 | best_score=56.884972 | best_active_count=20.0 | vol_eff=0.243951 | comp=5.307855e+00 | w=9.020407e-03
[00000] | Active Seeds/Total=20/20 | L_total=5.6885e+01 | L_vol=6.344e-02 L_fem=5.308e+00 L_wact=0.000e+00 L_active=0.000e+00 L_strut=0.000e+00 L_rep=0.000e+00 L_bnd=0.000e+00 |vol=0.488 vol_eff=0.244 (/0.150) tau=2.000e-02 os=1.00e+00 comp=5.308e+00 | hard_refine=off | w=9.020e-03 h=2.000e-01 | bw=1.000e-03 ba=1.000e+00 bb=3.000e-03 | theta=0.000e+00 a=0.000e+00 | Lse=0.000e+00 Lsv=0.000e+00 | rho(min/mean/max)=0.001/0.509/1.000 rho_b(min/mean/max)=0.000/0.215/0.999 rho_v(min/mean/max)=0.000/0.313/0.

Widget(value='<iframe src="http://localhost:33859/index.html?ui=P_0x7859203e82e0_14&reconnect=auto" class="pyv…

In [ ]:

threshold= 0.85
trainer.visualize_result_final(result, points_xyz, faces_ijk, thr=threshold, show_solid=False)
trainer.Visualize_fresult_final_fiber_Direction_3D( result, points_xyz,faces_ijk,thr=threshold)
out = trainer.visualize_result_final_fiber_direction_2d(result, points_xyz, faces_ijk,thr=threshold)
display(out["figure"])
trainer.visualize_best_seed_activity(result, points_xyz, faces_ijk)
#trainer.visualize_result_final_smooth_surface_pyvista(result, points_xyz, faces_ijk, thr=0.5)
#print(result["tensorboard_log_dir"])


In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
#Voronoi visualization
RUN_THIS = True
if  RUN_THIS:
    seeds_raw= torch.tensor([
        [-0.9, 0.20],
        [-0.9, 0.40],
        [-0.9, 0.60],
        [-0.9, 0.80],

        [0.3, 0.40],
        [0.9, 0.20],
        [0.9, 0.40],
        [0.9, 0.60],
        [0.9, 0.80],
        [0.3, 0.40],
        [0.9, 0.20],
        [0.9, 0.40],
        [0.9, 0.60],
        [0.9, 0.80],

    ], dtype=torch.float32)
    # S = 20
    # margin = 0
    # seeds_raw = margin + (1 - 2 * margin) * torch.rand((S, 2), dtype=torch.float32)
    #seeds_raw = generator.fps(uv, K=10, M=10, return_indices=False).cpu()  # FPS seeds from the input UV points
    # N= 15
    # seeds_raw_index= generator.fps_3d(points_xyz,N,boundary_idx_ring1)
    # seeds_raw = uv[seeds_raw_index]


    S = seeds_raw.shape[0]
    # symmetric pairwise widths
    w_raw = torch.full((S, S), 5 ,dtype=torch.float32)
    # w_raw = (torch.rand(S, S) * 40) - 20
    # w_raw =  0.5*(w_raw + w_raw.T)
    #print(w_raw)


    # optional anisotropy

    theta = 2 * torch.pi * torch.rand(S, dtype=torch.float32)
    a_raw = 3* torch.randn(S, dtype=torch.float32)



    face_tensor = tensors["face_tensors"][0]   # choose the face you want to test

    uv = face_tensor["uv"]
    Xu = face_tensor["Xu"]
    Xv = face_tensor["Xv"]
    points_xyz = face_tensor["points_xyz"]
    faces_ijk = face_tensor["faces_ijk"]

    Veron_viz = VoronoiModelVisualizer.from_face_tensor(
        face_tensor,
        tau=0.01,
        n_seeds=S,
        seed_domain_temp=0.02,
        seed_domain_mask_threshold=0.5,
        use_metric_anisotropy=False,
        beta=0.01,
        density_projection_strength=0.45,
        density_projection_threshold=0.35,
        density_projection_gamma=0.06,
        w_max_ratio=0.1,
        fixed_height=0.2,
        use_boundary_attachment=True,
        boundary_attach_width_min=1e-6,
        boundary_attach_width_max=2e-2,
        raw_temp=0.01,
    )
    result = Veron_viz.visualize_fields(
        seeds_raw=seeds_raw,
        w_raw=w_raw,
        theta=None,
        a_raw=None,
        boundary_width_raw=torch.tensor(1.0),
        boundary_alpha_raw=torch.tensor(1),
        boundary_beta_raw=torch.tensor(0.0),
        show_uv=True,
        show_3d=True,
        fiber_stride=1,
        fiber_scale_uv=0.05,
        fiber_scale_3d=0.5,
        fiber_min_strength=0.5,
        show_fiber_density_background=False,
        hard_seed_mask=False,
    )

    display(result["uv_fig"])
    result["plotter"].show()
